## J1 — EXPLAIN FORMATTED on gold join

```sql
EXPLAIN FORMATTED
SELECT dc.connector_type, SUM(fs.energy_kwh) AS total_energy
FROM   vp_rp20040126.gold.fact_sessions  fs
JOIN   vp_rp20040126.gold.dim_charger     dc  ON fs.charger_id = dc.charger_id
GROUP BY dc.connector_type;
```

The join operator may be BroadcastHashJoin if dim_charger is small, because Spark can broadcast it to avoid shuffling the large fact table. The Exchange operator indicates data shuffling between partitions. The actual join operator depends on the execution plan generated by EXPLAIN FORMATTED.

## J2 — COPY INTO vs Auto Loader

**When COPY INTO is the better choice:**

Use COPY INTO when you are doing scheduled, batch-style loads — ex a nightly job that picks up whatever files landed in a folder since the last run. It's simple, works with a single SQL statement, and you don't need streaming infrastructure. Good fit when files arrive in batches and the schema is stable.

**When Auto Loader is the better choice:**

Use Auto Loader when files arrive continuously or in very frequent small batches, when you need near-real-time freshness, or when the schema might evolve over time. It's built on Structured Streaming, so it handles file discovery efficiently (incremental listing instead of scanning the whole directory every time), supports schema inference and evolution, and works well with trigger function for available now = true for batch-style processing on top.



## J3 — Promoting the job to Prod with Declarative Automation Bundles
. To promote the job to Prod (catalog vp_prod ) with Declarative Automation
Bundles: name the config file, the section defining the job, how each environment gets
its own catalog, and the deploy command.


Config file -  databricks.yml — this sits at the bundle root and is the main entry point. It declares the bundle name, includes resource files, defines variables, and sets up per-environment targets.

resources and jobs -  in the current project the job lives in resources/pipe.yaml directory as an yaml files . also all the resource like jobs and other yaml file are available hare for the project bundle.

enviroment settings and configration - for the enviroment we can make the varibale of the env parameter which can be passed to the db.yaml or root file at deploying time and env get desided.
also for the jobs we create the catalog vaible and pass the parameter into the job while runnign it instead of harcoding the vatalog name.

Deploy cmd --

databricks bundle deploy -t enviroment_name

env name - dev , uat or prod

